In [1]:
!pip install transformers accelerate sentencepiece gradio torch --quiet


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

MODEL = "mistralai/Mistral-7B-Instruct-v0.2"   # PUBLIC, NO LOGIN REQUIRED

tokenizer = AutoTokenizer.from_pretrained(MODEL , use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Creative Writing Model Loaded!")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Creative Writing Model Loaded!


In [4]:
def generate_creative_writing(prompt, style, max_tokens):

    system_prompt = (
        "You are a master creative writer. Write vivid, emotional, imaginative prose with strong metaphors, "
        "beautiful language, sensory detail, rhythm, personality, and atmosphere. Avoid repetition. "
        f"Write in the style: {style}."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_tokens,
        temperature=0.95,   # more creativity
        top_p=0.95,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
def app_interface(prompt, style, max_tokens):
    if not prompt:
        return "Please enter a prompt."
    return generate_creative_writing(prompt, style, max_tokens)

demo = gr.Interface(
    fn=app_interface,
    inputs=[
        gr.Textbox(lines=4, label="Your Idea / Prompt"),
        gr.Dropdown(
            ["Poetic", "Fantasy", "Sci-Fi", "Romantic", "Dark & Atmospheric", "Cinematic", "Emotional"],
            label="Writing Style"
        ),
        gr.Slider(50, 500, value=250, label="Max Tokens")
    ],
    outputs=gr.Textbox(lines=12, label="Creative Writing Output"),
    title="🎨 Creative Writing Generator (Decoder-Only, No Login)",
    description="Uses Mistral-7B-Instruct to generate imaginative, artistic writing."
)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4957239f516f794d65.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
